In [11]:
# test some lag correlation between kelp
import pandas as pd
import numpy as np

d = pd.read_csv("../../1_DATA/processed/midcal/midcal_kelp_sst_labeled.csv",
                index_col=0, parse_dates=True).sort_index()

print("columns:", list(d.columns))

# pick columns robustly
def pick_col(cands):
    for c in cands:
        if c in d.columns:
            return c
    return None

kelp_col = pick_col(["kelp_smooth","kelp","kelp_canopy","kelp_area","canopy"])
sst_col  = pick_col(["sst_anom","sstanom","sst","sst_q_mean","sstanom_q_mean"])

if kelp_col is None:
    raise ValueError("No kelp column found. Check printed columns.")
if sst_col is None:
    raise ValueError("No SST/anom column found. Check printed columns.")

# optional: focus on anomalies if you have both
if "sst_anom" in d.columns:
    sst_col = "sst_anom"

# Kelp change (quarter-to-quarter)
d["dK"] = d[kelp_col].diff()

def corr_at_lags(x, y, max_lag=8, method="pearson"):
    out = []
    for L in range(0, max_lag+1):
        # y.shift(L) = SST earlier -> kelp later
        out.append((L, x.corr(y.shift(L), method=method)))
    return pd.DataFrame(out, columns=["lag_quarters", f"corr_{method}"]).set_index("lag_quarters")

print("\nCorr(kelp level vs SST) by lag:")
print(corr_at_lags(d[kelp_col], d[sst_col], max_lag=8))

print("\nCorr(Δkelp vs SST) by lag (often more meaningful):")
print(corr_at_lags(d["dK"], d[sst_col], max_lag=8))

# Era split (edit date if your big break is different)
cut = pd.Timestamp("2014-01-01")
pre  = d.loc[:cut].dropna(subset=[kelp_col, sst_col])
post = d.loc[cut:].dropna(subset=[kelp_col, sst_col])

print("\nPearson corr (pre-2014):", pre[kelp_col].corr(pre[sst_col]))
print("Pearson corr (post-2014):", post[kelp_col].corr(post[sst_col]))

columns: ['kelp_area', 'kelp_smooth', 'coverage', 'coverage_frac', 'kelp_q_z', 'kelp_z_1yr', 'collapse', 'suppressed', 'sstanom_q_mean', 'sstanom_q_max', 'sstanom_q_mean_lag1']

Corr(kelp level vs SST) by lag:
              corr_pearson
lag_quarters              
0                 0.095995
1                 0.096313
2                 0.124779
3                 0.147056
4                 0.222787
5                 0.239338
6                 0.203722
7                 0.179638
8                 0.134889

Corr(Δkelp vs SST) by lag (often more meaningful):
              corr_pearson
lag_quarters              
0                -0.110938
1                -0.008437
2                 0.059953
3                 0.045263
4                 0.161901
5                 0.029788
6                -0.072691
7                -0.071848
8                -0.117510

Pearson corr (pre-2014): -0.013327659834917297
Pearson corr (post-2014): -0.010588492723991566


In [12]:
# midcal quarterly SST + heat-stress features
# - prints full table (or saves it)
# - makes plots + saves to 5_FIGURES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# -----------------------
# INPUT / OUTPUT PATHS
# -----------------------
sst_m_path = Path("../../1_DATA/processed/midcal/oisst_midcal_bbox_monthly.csv")
out_csv    = Path("../../1_DATA/processed/midcal/midcal_sst_heatstress_quarterly.csv")

fig_dir = Path("../../5_FIGURES")
fig_dir.mkdir(parents=True, exist_ok=True)

# -----------------------
# LOAD MONTHLY SST
# -----------------------
sst_m = pd.read_csv(sst_m_path, parse_dates=["time"]).set_index("time").sort_index()

# -----------------------
# BUILD HEAT-STRESS FEATURES
# -----------------------
CLIM_START, CLIM_END = "1991-01-01", "2020-12-31"
base = sst_m.loc[CLIM_START:CLIM_END, "sst_anom"]

# month-specific 90th percentile threshold
p90_by_month = base.groupby(base.index.month).quantile(0.90)

sst_m["anom_p90"] = sst_m.index.month.map(p90_by_month)
sst_m["anom_exceed"] = (sst_m["sst_anom"] > sst_m["anom_p90"]).astype(int)
sst_m["anom_pos"] = sst_m["sst_anom"].clip(lower=0)

# quarterly (quarter-start) aggregation
q = sst_m.resample("QS")

feat = pd.DataFrame({
    "sst_q_mean": q["sst"].mean(),
    "anom_q_mean": q["sst_anom"].mean(),
    "anom_q_max": q["sst_anom"].max(),
    "anom_pos_sum": q["anom_pos"].sum(),             # cumulative positive anomaly (heat stress proxy)
    "anom_exceed_months": q["anom_exceed"].sum(),    # count months above p90 threshold
})

# lags (1 quarter)
feat["anom_pos_sum_lag1"] = feat["anom_pos_sum"].shift(1)
feat["anom_q_max_lag1"]   = feat["anom_q_max"].shift(1)

# -----------------------
# VIEW / SAVE FULL TABLE
# -----------------------
print("Rows:", len(feat))
print("Time range:", feat.index.min(), "->", feat.index.max())

# Save full table (recommended)
out_csv.parent.mkdir(parents=True, exist_ok=True)
feat.to_csv(out_csv, index=True)
print("Saved full table to:", out_csv.resolve())

# Print the whole thing (can be long in notebook; comment out if annoying)
print("\nFULL TABLE:")
print(feat.to_string())

print("\nLAST 12 rows:")
print(feat.tail(12))

# -----------------------
# PLOTS (SAVED)
# -----------------------
plots = [
    (["sst_q_mean"], "midcal_sst_q_mean.png", "NorCal SST (quarterly mean)", "SST (°C)"),
    (["anom_q_mean", "anom_q_max"], "midcal_sst_anom_mean_max.png",
     "NorCal SST anomaly (quarterly mean and quarterly max)", "Anomaly (°C)"),
    (["anom_pos_sum"], "midcal_heatstress_possum.png",
     "NorCal heat stress: sum of positive anomalies per quarter", "°C-months (proxy)"),
    (["anom_exceed_months"], "midcal_heatstress_exceed_months.png",
     "NorCal extreme heat months per quarter (p90 threshold)", "Months (count)"),
]

for cols, fname, title, ylabel in plots:
    fig, ax = plt.subplots(figsize=(12, 4))
    for c in cols:
        ax.plot(feat.index, feat[c], label=c)
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.legend()
    fig.tight_layout()
    outpath = fig_dir / fname
    fig.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("saved plot:", outpath.resolve())

Rows: 179
Time range: 1981-07-01 00:00:00 -> 2026-01-01 00:00:00
Saved full table to: /Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/midcal_sst_heatstress_quarterly.csv

FULL TABLE:
            sst_q_mean  anom_q_mean  anom_q_max  anom_pos_sum  anom_exceed_months  anom_pos_sum_lag1  anom_q_max_lag1
time                                                                                                                 
1981-07-01   13.837731    -1.739406   -1.739406      0.000000                   0                NaN              NaN
1981-10-01   13.976926    -0.066066    0.439978      0.631658                   0           0.000000        -1.739406
1982-01-01   12.324566    -0.180765    0.083435      0.083435                   0           0.631658         0.439978
1982-04-01   12.898144     0.012357    0.489971      0.759934                   0           0.083435         0.083435
1982-07-01   14.467238    -0.642728   -0.346110      0.000000                   0           0.7

In [13]:
# TEST 1: Do SST anomaly peaks precede "suppressed" kelp?
# Outputs:
# 1) AUC vs lag table + plot
# 2) mean anomaly-peak difference table (suppressed - not)
# 3) overlay plot: lagged SST peak vs suppressed label over time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score

# -----------------------
# LOAD DATA
# -----------------------
DATA_PATH = Path("../../1_DATA/processed/midcal/midcal_kelp_sst_labeled.csv")
d = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True).sort_index()

# Required columns check
required = ["suppressed", "sstanom_q_max"]
missing = [c for c in required if c not in d.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Found columns: {list(d.columns)}")

# Align index to quarter-start (important for correct lag interpretation)
qidx = d.index.to_period("Q").to_timestamp(how="start")
d = d.copy()
d.index = qidx

# Labels and predictor
y = d["suppressed"].astype(int)
x0 = d["sstanom_q_max"]  # peak anomaly per quarter

print("Loaded:", DATA_PATH.resolve())
print("Rows:", len(d), "Time:", d.index.min(), "->", d.index.max())
print("Suppressed counts:", y.value_counts().to_dict())

# -----------------------
# CORE METRICS BY LAG
# -----------------------
MAX_LAG = 8  # quarters
rows = []

for L in range(0, MAX_LAG + 1):
    # SST earlier -> label later
    x = x0.shift(L)

    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    if tmp["y"].nunique() < 2:
        continue

    auc = roc_auc_score(tmp["y"], tmp["x"])

    g0 = tmp[tmp["y"] == 0]["x"]
    g1 = tmp[tmp["y"] == 1]["x"]
    mean0 = g0.mean()
    mean1 = g1.mean()
    diff = mean1 - mean0

    rows.append({
        "lag_quarters": L,
        "n": len(tmp),
        "auc": auc,
        "mean_x_not_supp": mean0,
        "mean_x_supp": mean1,
        "mean_diff_supp_minus_not": diff,
    })

res = pd.DataFrame(rows).set_index("lag_quarters")
print("\n=== Results (SST peak vs suppressed) ===")
print(res)

# Best lag by AUC
best_lag = res["auc"].idxmax()
print(f"\nBEST LAG = {best_lag} quarters  (AUC={res.loc[best_lag,'auc']:.3f}, n={int(res.loc[best_lag,'n'])})")

# -----------------------
# PLOTS (SAVED)
# -----------------------
FIG_DIR = Path("../../5_FIGURES")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Plot 1: AUC vs lag
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(res.index, res["auc"], marker="o")
ax.axhline(0.5, linestyle="--")
ax.set_title("Predict suppressed using SST anomaly max (AUC vs lag)")
ax.set_xlabel("Lag (quarters) — SST earlier → suppressed later")
ax.set_ylabel("AUC (0.5=random, 1.0=perfect)")
fig.tight_layout()

out1 = FIG_DIR / "midcal_auc_suppressed_sstanom_q_max.png"
fig.savefig(out1, dpi=200, bbox_inches="tight")
plt.close(fig)
print("saved:", out1.resolve())

# Plot 2: Time series overlay (best lag)
x_best = x0.shift(best_lag)

# For visualization, make a 0/1 line for suppressed and the SST anomaly peaks
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(d.index, x_best, label=f"sstanom_q_max (lag {best_lag}q)")
ax.plot(d.index, y, label="suppressed (0/1)", alpha=0.8)

ax.set_title(f"Suppressed kelp vs lagged SST anomaly peaks (lag={best_lag} quarters)")
ax.set_xlabel("Year (quarter start)")
ax.set_ylabel("SST anomaly (°C) / label")
ax.legend()
fig.tight_layout()

out2 = FIG_DIR / f"midcal_overlay_suppressed_vs_sstanom_q_max_lag{best_lag}.png"
fig.savefig(out2, dpi=200, bbox_inches="tight")
plt.close(fig)
print("saved:", out2.resolve())

# Optional: show the best-lag boxplot (very interpretable)
tmp = pd.DataFrame({"x": x_best, "y": y}).dropna()
g0 = tmp[tmp["y"] == 0]["x"].values
g1 = tmp[tmp["y"] == 1]["x"].values

fig, ax = plt.subplots(figsize=(6,4))
ax.boxplot([g0, g1], tick_labels=["not suppressed", "suppressed"])
ax.set_title(f"SST anomaly peak (lag {best_lag}q) by kelp state")
ax.set_ylabel("sstanom_q_max (°C)")
fig.tight_layout()

out3 = FIG_DIR / f"midcal_box_sstanom_q_max_lag{best_lag}_by_suppressed.png"
fig.savefig(out3, dpi=200, bbox_inches="tight")
plt.close(fig)
print("saved:", out3.resolve())

Loaded: /Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/midcal_kelp_sst_labeled.csv
Rows: 157 Time: 1984-04-01 00:00:00 -> 2025-07-01 00:00:00
Suppressed counts: {0: 140, 1: 17}

=== Results (SST peak vs suppressed) ===
                n       auc  mean_x_not_supp  mean_x_supp  \
lag_quarters                                                
0             157  0.494118         0.435096     0.376739   
1             156  0.535760         0.415871     0.459925   
2             155  0.642796         0.378833     0.766354   
3             154  0.727351         0.352985     0.989776   
4             153  0.697232         0.362175     0.931968   
5             152  0.684532         0.368889     0.867703   
6             151  0.652766         0.384281     0.757625   
7             150  0.518355         0.420507     0.424524   
8             149  0.481283         0.423377     0.337257   

              mean_diff_supp_minus_not  
lag_quarters                            
0           

In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

d = pd.read_csv("../../1_DATA/processed/midcal_kelp_sst_labeled.csv",
                index_col=0, parse_dates=True).sort_index()

# align to quarter start
d.index = d.index.to_period("Q").to_timestamp(how="start")

lag = 4
x = d["sstanom_q_max"].shift(lag)          # SST peak earlier by 4 quarters
y = d["suppressed"].astype(int)

tmp = pd.DataFrame({"x": x, "y": y}).dropna()

g0 = tmp[tmp.y == 0].x.values
g1 = tmp[tmp.y == 1].x.values

auc = roc_auc_score(tmp.y, tmp.x)

mean0, mean1 = g0.mean(), g1.mean()
diff = mean1 - mean0
pooled = np.sqrt(((len(g0)-1)*g0.var(ddof=1) + (len(g1)-1)*g1.var(ddof=1)) / (len(g0)+len(g1)-2))
d_cohen = diff / pooled

print("lag:", lag, "quarters")
print("n not suppressed:", len(g0), "mean:", mean0)
print("n suppressed    :", len(g1), "mean:", mean1)
print("mean diff (supp - not):", diff)
print("Cohen's d:", d_cohen)
print("AUC:", auc)

lag: 4 quarters
n not suppressed: 136 mean: 0.3621752054036764
n suppressed    : 17 mean: 0.9319680727647058
mean diff (supp - not): 0.5697928673610294
Cohen's d: 0.7151778423190955
AUC: 0.6972318339100345


In [16]:
# --- TIME-SERIES-SAFE SIGNIFICANCE: block bootstrap CI ---
from sklearn.metrics import roc_auc_score
import numpy as np

def block_bootstrap_ci(x, y, block_len=4, B=3000, seed=0):
    x = np.asarray(x)
    y = np.asarray(y).astype(int)
    n = len(x)

    starts = np.arange(0, n - block_len + 1)
    n_blocks = int(np.ceil(n / block_len))
    rng = np.random.default_rng(seed)

    diffs, aucs = [], []
    for _ in range(B):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        idx = np.concatenate([np.arange(s, s + block_len) for s in chosen])[:n]

        xb, yb = x[idx], y[idx]
        if yb.min() == yb.max():  # skip degenerate resample
            continue

        diffs.append(xb[yb == 1].mean() - xb[yb == 0].mean())
        aucs.append(roc_auc_score(yb, xb))

    diffs = np.array(diffs)
    aucs = np.array(aucs)

    diff_ci = np.quantile(diffs, [0.025, 0.975])
    auc_ci  = np.quantile(aucs,  [0.025, 0.975])

    return diff_ci, auc_ci, diffs.mean(), aucs.mean(), len(diffs)

# Use tmp (already dropna'd)
diff_ci, auc_ci, diff_mean_bs, auc_mean_bs, n_ok = block_bootstrap_ci(
    tmp["x"].values, tmp["y"].values,
    block_len=4, B=3000, seed=42
)

print("\nBLOCK BOOTSTRAP (block_len=4 quarters)")
print("resamples used:", n_ok)
print("mean diff bootstrap mean:", diff_mean_bs, "95% CI:", diff_ci)
print("AUC bootstrap mean      :", auc_mean_bs, "95% CI:", auc_ci)

# quick conclusion
print("\nConclusion:")
print("Mean diff significant?" , (diff_ci[0] > 0) or (diff_ci[1] < 0))
print("AUC significant?"       , (auc_ci[0] > 0.5) or (auc_ci[1] < 0.5))


BLOCK BOOTSTRAP (block_len=4 quarters)
resamples used: 2998
mean diff bootstrap mean: 0.5925799252962357 95% CI: [0.16185256 1.10626452]
AUC bootstrap mean      : 0.7014373886407874 95% CI: [0.54127758 0.85432798]

Conclusion:
Mean diff significant? True
AUC significant? True


In [15]:
import pandas as pd

# ERDDAP CSV has a 2nd row of units -> skiprows=[1]
url = (
    "https://coastwatch.pfeg.noaa.gov/erddap/griddap/erdUI39mo.csv"
    "?upwelling_index[(1984-01-15T00:00:00Z):1:(2025-12-15T00:00:00Z)][(39.0)][(235.0)],"
    "upwelling_index_anomaly[(1984-01-15T00:00:00Z):1:(2025-12-15T00:00:00Z)][(39.0)][(235.0)]"
)

ui = pd.read_csv(url, skiprows=[1])
ui["time"] = pd.to_datetime(ui["time"])
ui = ui.set_index("time").sort_index()

# align monthly timestamps to month-start for easier joins
ui.index = ui.index.to_period("M").to_timestamp(how="start")

ui_q = ui.resample("QS").agg({
    "upwelling_index": "mean",
    "upwelling_index_anomaly": "mean",
})
ui_q["upwelling_index_lag1"] = ui_q["upwelling_index"].shift(1)

print(ui_q.head())

            upwelling_index  upwelling_index_anomaly  upwelling_index_lag1
time                                                                      
1984-01-01        15.333333                 4.666667                   NaN
1984-04-01       197.333333                77.333333             15.333333
1984-07-01       213.333333                85.333333            197.333333
1984-10-01         2.333333                 2.000000            213.333333
1985-01-01        43.333333                32.666667              2.333333


/var/folders/61/3wm_gk5j5jvd7cv3mv31_8940000gn/T/ipykernel_49897/908861498.py:15: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ui.index = ui.index.to_period("M").to_timestamp(how="start")
